### NBA Data Mining Project

By Samuel Scott and Yubi Joy Quinzon

Project Objective: Develop a classification model that predicts whether an NBA player will score over or under a
predefined point threshold in a given game. The model leverages historical player performance
data, opponent defensive metrics, and contextual game features to estimate the likelihood of
exceeding the threshold.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as stat
import seaborn as sns
import math
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression #Could use Random Forest or other classification models
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

In [2]:
nba = pd.read_csv('C:/Users/samsc/Desktop/ADS-502/NBA_Datasets/PlayerStatistics.csv')

In [3]:
nba.head()

,firstName,lastName,personId,gameId,gameDateTimeEst,playerteamCity,playerteamName,opponentteamCity,opponentteamName,gameType,...,threePointersPercentage,freeThrowsAttempted,freeThrowsMade,freeThrowsPercentage,reboundsDefensive,reboundsOffensive,reboundsTotal,foulsPersonal,turnovers,plusMinusPoints
0,Luke,Kennard,1628379.0,22501098,2026-03-30 22:00:00,Los Angeles,Lakers,Washington,Wizards,Regular Season,...,0.8,1.0,1.0,1.0,1.0,1.0,2.0,3.0,1.0,20.0
1,Maxi,Kleber,1628467.0,22501098,2026-03-30 22:00:00,Los Angeles,Lakers,Washington,Wizards,Regular Season,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Jarred,Vanderbilt,1629020.0,22501098,2026-03-30 22:00:00,Los Angeles,Lakers,Washington,Wizards,Regular Season,...,0.0,4.0,2.0,0.5,6.0,2.0,8.0,0.0,1.0,5.0
3,Deandre,Ayton,1629028.0,22501098,2026-03-30 22:00:00,Los Angeles,Lakers,Washington,Wizards,Regular Season,...,0.0,2.0,2.0,1.0,5.0,2.0,7.0,0.0,2.0,9.0
4,Luka,Doncic,1629029.0,22501098,2026-03-30 22:00:00,Los Angeles,Lakers,Washington,Wizards,Regular Season,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
nba.columns  #Not all columns are needed to answer the project objective we should subset this dataframe

Index(['firstName', 'lastName', 'personId', 'gameId', 'gameDateTimeEst',
       'playerteamCity', 'playerteamName', 'opponentteamCity',
       'opponentteamName', 'gameType', 'gameLabel', 'gameSubLabel',
       'seriesGameNumber', 'win', 'home', 'numMinutes', 'points', 'assists',
       'blocks', 'steals', 'fieldGoalsAttempted', 'fieldGoalsMade',
       'fieldGoalsPercentage', 'threePointersAttempted', 'threePointersMade',
       'threePointersPercentage', 'freeThrowsAttempted', 'freeThrowsMade',
       'freeThrowsPercentage', 'reboundsDefensive', 'reboundsOffensive',
       'reboundsTotal', 'foulsPersonal', 'turnovers', 'plusMinusPoints'],
      dtype='object')

In [5]:
nba.shape #1,664,775 rows and 35 columns

(1664775, 35)

In [6]:
nba.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1664775 entries, 0 to 1664774
Data columns (total 35 columns):
 #   Column                   Non-Null Count    Dtype  
---  ------                   --------------    -----  
 0   firstName                1664541 non-null  object 
 1   lastName                 1664537 non-null  object 
 2   personId                 1664730 non-null  float64
 3   gameId                   1664775 non-null  int64  
 4   gameDateTimeEst          1661047 non-null  object 
 5   playerteamCity           1664607 non-null  object 
 6   playerteamName           1661047 non-null  object 
 7   opponentteamCity         1660879 non-null  object 
 8   opponentteamName         1661047 non-null  object 
 9   gameType                 1664725 non-null  object 
 10  gameLabel                96782 non-null    object 
 11  gameSubLabel             6497 non-null     object 
 12  seriesGameNumber         135048 non-null   float64
 13  win                      1661037 non-null 

Some of the columns might not be necessary to do our analysis. For example fieldGoalsAttempted, fieldGoalsMade, and fieldGoalsPercentage are all 
similar and it would be arguable that we just need fieldGoalsPercentage. Reducing dimensionality is good to make data less complicated.

In [7]:
nba.isnull().sum().sort_values() #Shows all the missing data in our dataset, this gives us a better idea of which columns to drop

gameId                           0
personId                        45
gameType                        50
playerteamCity                 168
firstName                      234
lastName                       238
threePointersPercentage       1718
threePointersMade             1718
threePointersAttempted        1718
fieldGoalsPercentage          1718
fieldGoalsMade                1718
fieldGoalsAttempted           1718
steals                        1718
blocks                        1718
assists                       1718
points                        1718
freeThrowsPercentage          1718
reboundsDefensive             1718
freeThrowsAttempted           1718
freeThrowsMade                1718
reboundsOffensive             1718
reboundsTotal                 1718
turnovers                     1718
foulsPersonal                 1718
plusMinusPoints               2666
opponentteamName              3728
gameDateTimeEst               3728
playerteamName                3728
home                

In [8]:
#These columns are being dropped because had a large amount of missing data
dropped_columns = ['seriesGameNumber', 'gameLabel', 'gameSubLabel', 'numMinutes']
nba = nba.drop(columns=dropped_columns)

In [9]:
nba.columns #checked to see if dropped columns are removed from NBA dataset
nba.isnull().sum().sort_values()

Index(['firstName', 'lastName', 'personId', 'gameId', 'gameDateTimeEst',
       'playerteamCity', 'playerteamName', 'opponentteamCity',
       'opponentteamName', 'gameType', 'win', 'home', 'points', 'assists',
       'blocks', 'steals', 'fieldGoalsAttempted', 'fieldGoalsMade',
       'fieldGoalsPercentage', 'threePointersAttempted', 'threePointersMade',
       'threePointersPercentage', 'freeThrowsAttempted', 'freeThrowsMade',
       'freeThrowsPercentage', 'reboundsDefensive', 'reboundsOffensive',
       'reboundsTotal', 'foulsPersonal', 'turnovers', 'plusMinusPoints'],
      dtype='object')

personId, gameType, playerTeamCity, firstName, lastName are all categorical data that is missing. These missing values will 
be labled Unknown since they are not quantifiable attributes. personId and gameType will be treated differently.

In [11]:
#Drop IDs that are missing because we need to be able to identify NBA player. 
#Game type is important to determine a player's scoring so we need that information too so we drop rows where they are missing.
nba = nba.dropna(subset=['personId', 'gameType'])

In [12]:
#Fill NA values with Unknown
nba['firstName'] = nba['firstName'].fillna('Unknown')
nba['lastName'] = nba['lastName'].fillna('Unknown')
nba['playerteamCity'] = nba['playerteamCity'].fillna('Unknown')

The best approach to deal with missing data in our continuous variables might be to impute the statistical categories. This is because there are a lot of rows in this dataset and imputing the data, allows us to get a rough estimate on how much a player would score. 

In [17]:
#Testing to see what measure would best suit the data to impute
a=nba['assists'].mean()
b=nba['assists'].median()
c=nba['assists'].mode()
print(a)
print(b)
print(c)

d=nba['blocks'].mean()
e=nba['blocks'].median()
f=nba['blocks'].mode()
print(d)
print(e)
print(f)

g=nba['threePointersMade'].mean()
h=nba['threePointersMade'].median()
i=nba['threePointersMade'].mode()
print(g)
print(h)
print(i)

j=nba['threePointersPercentage'].mean()
k=nba['threePointersPercentage'].median()
l=nba['threePointersPercentage'].mode()
print(j)
print(k)
print(l)

1.9606797990573446
1.0
0    0.0
Name: assists, dtype: float64
0.36838905519188053
0.0
0    0.0
Name: blocks, dtype: float64
0.43732568753826007
0.0
0    0.0
Name: threePointersMade, dtype: float64
0.11756383806114709
0.0
0    0.0
Name: threePointersPercentage, dtype: float64


In [18]:
nba.columns

Index(['firstName', 'lastName', 'personId', 'gameId', 'gameDateTimeEst',
       'playerteamCity', 'playerteamName', 'opponentteamCity',
       'opponentteamName', 'gameType', 'win', 'home', 'points', 'assists',
       'blocks', 'steals', 'fieldGoalsAttempted', 'fieldGoalsMade',
       'fieldGoalsPercentage', 'threePointersAttempted', 'threePointersMade',
       'threePointersPercentage', 'freeThrowsAttempted', 'freeThrowsMade',
       'freeThrowsPercentage', 'reboundsDefensive', 'reboundsOffensive',
       'reboundsTotal', 'foulsPersonal', 'turnovers', 'plusMinusPoints'],
      dtype='object')

Due to the ease of having a round number, imputing with the median might be the best aggregation measure to impute with. 

In [ ]:
#Considering dropping more columns for sure but needs discussion 
continuous_vars = ['threePointersPercentage', 'threePointersAttempted', 'threePointersMade', 'freeThrowsAttempted',
                   'freeThrowsMade', 'freeThrowsPercentage', 'reboundsDefensive', 'reboundsOffensive', 'reboundsTotal', 
                   'assists', 'blocks', 'steals', 'turnovers']